# 11 · Gold · Tabelas por assunto
Tabelas prontas para o dashboard (Henry) e para o agente (Emanuel), lidas da silver unificada (notebook 10).

- **Só vendas válidas:** o filtro `WHERE valido` acontece aqui, e só aqui (D07).
- **Uma tabela por pergunta de negócio, sempre com a coluna `canal`.** O site entra sem mudar as colunas.
- **Proposta para alinhar** (quadro "Fluxo de dados e tabelas gold"): as colunas podem mudar depois da conversa com o Henry e o Emanuel.

| Tabela | Para quê | Grão |
|---|---|---|
| `gold_vendas_diarias` | Receita, unidades e margem | dia × canal × loja × categoria |
| `gold_pedidos_diarios` | Ticket médio | dia × canal × loja |
| `gold_estoque_diario` | Giro de estoque e ruptura | dia × canal × loja × grupo |
| `gold_catalogo` | Agente de recomendação (D21) | produto |

**Ficam para depois do site:** `gold_funil_diario` (conversão e abandono) e `gold_avaliacoes`.

**Equivalente no BigQuery:** `DECIMAL` → `NUMERIC`; `COUNT_IF` → `COUNTIF`; `bool_or` → `LOGICAL_OR`; `date_sub(d, 30)` → `DATE_SUB(d, INTERVAL 30 DAY)`. Em produção, as tabelas seriam particionadas por `data` e clusterizadas por `canal`.

### 1. `gold_vendas_diarias`
Margem bruta = receita − custo. No marketplace o custo é **estimado** pela margem da categoria (D20): `custo_estimado = true` precisa aparecer no dashboard.

In [0]:
CREATE OR REPLACE TABLE horizonte.gold.gold_vendas_diarias AS
SELECT data, canal, loja_id, categoria, grupo,
       SUM(quantidade)            AS unidades,
       SUM(receita)               AS receita,
       SUM(custo)                 AS custo,
       SUM(receita) - SUM(custo)  AS margem_bruta,
       bool_or(custo_estimado)    AS custo_estimado
FROM horizonte.silver.fato_vendas
WHERE valido
GROUP BY data, canal, loja_id, categoria, grupo;

### 2. `gold_pedidos_diarios`
Tabela própria porque um pedido com produtos de duas categorias aparece em duas linhas da tabela de vendas.
**Ticket médio de um período = soma da receita ÷ soma dos pedidos.** Nunca a média dos tickets diários.

In [0]:
CREATE OR REPLACE TABLE horizonte.gold.gold_pedidos_diarios AS
SELECT data, canal, loja_id,
       COUNT(DISTINCT pedido_id)                                        AS pedidos,
       SUM(receita)                                                     AS receita,
       CAST(SUM(receita) / COUNT(DISTINCT pedido_id) AS DECIMAL(12,2))  AS ticket_medio
FROM horizonte.silver.fato_vendas
WHERE valido
GROUP BY data, canal, loja_id;

### 3. `gold_estoque_diario`
Posição de estoque somada por grupo, a partir da movimentação diária por SKU (`silver.erp_estoque_diario`). Hoje só a loja física; o site entra com o gerador do site.
- `unidades_vendidas` são as **saídas do estoque**: incluem os itens das linhas de venda com sujeira (SKU nulo ou quantidade negativa), que foram vendidos de fato, mas cujo registro chegou corrompido. Por isso a soma fica 800 unidades acima das vendas válidas.
- **Giro de estoque** (proposta, a confirmar com o Henry): unidades vendidas no período ÷ estoque final médio no período.

In [0]:
CREATE OR REPLACE TABLE horizonte.gold.gold_estoque_diario AS
SELECT e.data,
       'loja_fisica'                  AS canal,
       e.loja_id,
       p.grupo,
       COUNT(*)                       AS skus,
       SUM(e.estoque_final)           AS estoque_final,
       SUM(e.entradas)                AS entradas,
       SUM(e.saidas)                  AS unidades_vendidas,
       COUNT_IF(e.estoque_final = 0)  AS skus_em_ruptura
FROM horizonte.silver.erp_estoque_diario e
JOIN horizonte.silver.dim_produto p ON p.sku = e.sku
GROUP BY e.data, e.loja_id, p.grupo;

### 4. `gold_catalogo`
Uma linha por produto, para o agente do Emanuel (D21).
- **Sem `custo` e sem `popularidade`:** são dados internos, e o agente conversa com clientes.
- **Unidades vendidas:** loja física e site (o marketplace não tem SKU do catálogo, D12). `unidades_30_dias` conta os 30 dias até a venda mais recente.
- **Avaliações:** `nota_media` e `qtd_avaliacoes` ficam vazias até o site existir.

In [0]:
CREATE OR REPLACE TABLE horizonte.gold.gold_catalogo AS
WITH ultima AS (
  SELECT MAX(data) AS dia
  FROM horizonte.silver.fato_vendas
  WHERE valido AND canal IN ('loja_fisica', 'site')
),
vendas AS (
  SELECT f.produto_id                                                         AS sku,
         SUM(f.quantidade)                                                    AS unidades_vendidas,
         SUM(CASE WHEN f.data > date_sub(u.dia, 30) THEN f.quantidade ELSE 0 END) AS unidades_30_dias
  FROM horizonte.silver.fato_vendas f
  CROSS JOIN ultima u
  WHERE f.valido AND f.canal IN ('loja_fisica', 'site')
  GROUP BY f.produto_id
)
SELECT p.sku, p.nome, p.modelo, p.categoria, p.grupo, p.tipo, p.variacao, p.cor,
       p.preco, p.descricao, p.ocasioes, p.imagem_url,
       CAST(NULL AS DECIMAL(3,2))        AS nota_media,       -- chega com as avaliações do site
       0                                 AS qtd_avaliacoes,
       COALESCE(v.unidades_vendidas, 0)  AS unidades_vendidas,
       COALESCE(v.unidades_30_dias, 0)   AS unidades_30_dias
FROM horizonte.silver.dim_produto p
LEFT JOIN vendas v ON v.sku = p.sku;

### 5. Conferência
Os totais da gold precisam bater com a silver: nenhuma venda pode se perder ou se duplicar na agregação.

In [0]:
SELECT v.canal,
       v.unidades,                                    -- loja_fisica 403.365 · marketplace 42.758
       v.receita,                                     -- loja_fisica 30.795.090,78 · marketplace 5.250.663,26
       p.pedidos,                                     -- loja_fisica 225.151
       ROUND(p.receita / p.pedidos, 2) AS ticket_medio  -- loja_fisica 136,78
FROM (SELECT canal, SUM(unidades) AS unidades, SUM(receita) AS receita
      FROM horizonte.gold.gold_vendas_diarias GROUP BY canal) v
JOIN (SELECT canal, SUM(pedidos) AS pedidos, SUM(receita) AS receita
      FROM horizonte.gold.gold_pedidos_diarios GROUP BY canal) p USING (canal)
ORDER BY v.canal;

In [0]:
SELECT
  (SELECT SUM(unidades_vendidas) FROM horizonte.gold.gold_estoque_diario)                      AS saidas_estoque,      -- 404.165
  (SELECT SUM(ABS(quantidade)) FROM horizonte.silver.erp_vendas WHERE NOT duplicada)           AS itens_vendidos_erp,  -- 404.165 (inclui as linhas com sujeira)
  (SELECT SUM(skus_em_ruptura) FROM horizonte.gold.gold_estoque_diario)                        AS loja_sku_dias_zerados; -- 19.560

In [0]:
SELECT COUNT(*)                                          AS produtos,             -- 200
       SUM(unidades_vendidas)                            AS unidades_vendidas,    -- 403.365 (só loja física por enquanto)
       COUNT_IF(grupo = 'Casa' AND unidades_vendidas > 0) AS casa_com_venda,      -- 0 (Casa só no online, D16)
       COUNT_IF(descricao IS NULL OR ocasioes IS NULL)   AS sem_descricao         -- 0
FROM horizonte.gold.gold_catalogo;